In [1]:
# --- Step 1: Clean and Install Working RDKit Build ---
!pip install rdkit --quiet
!pip install scikit-learn tqdm joblib --quiet

# Validate RDKit
from rdkit import Chem
from rdkit.Chem import AllChem
print("✅ RDKit imported successfully.")


✅ RDKit imported successfully.


In [2]:
import pandas as pd
ddi_df = pd.read_feather("ddi_rulesF.feather")
ddi_df.shape

(2855848, 2)

In [3]:
ddi_df.head()

,drug_a_id,drug_b_id
0,DB00001,DB06605
1,DB00001,DB06695
2,DB00001,DB01254
3,DB00001,DB01609
4,DB00001,DB01586


In [4]:
# --- Step 2: Load & Preprocess Datasets ---

import pandas as pd

# Load drug info and DDI rules
drugs_df = pd.read_csv("drugs.csv")
#ddi_df = pd.read_csv("ddi_rules_clean.csv")

# Keep only drugs with SMILES
usable_drugs = drugs_df[drugs_df['smiles'].notna()][['drugbank_id','smiles']]
usable_drugs.set_index('drugbank_id', inplace=True)

print(f"Total Drugs in Database: {len(drugs_df)}")
print(f"Usable Drugs with SMILES: {len(usable_drugs)} ({len(usable_drugs)/len(drugs_df):.2%})")

# Filter DDIs so both drugs have SMILES
ddi_df = ddi_df[
    ddi_df['drug_a_id'].isin(usable_drugs.index) &
    ddi_df['drug_b_id'].isin(usable_drugs.index)
].drop_duplicates(subset=['drug_a_id', 'drug_b_id'])

print(f"Filtered DDI pairs available for training: {len(ddi_df)}")


Total Drugs in Database: 17430
Usable Drugs with SMILES: 12313 (70.64%)
Filtered DDI pairs available for training: 2323786


In [5]:
# --- Step 3: Generate RDKit Fingerprints ---

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from tqdm import tqdm

from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

# Create the generator once
morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)

def mol_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = morgan_gen.GetFingerprint(mol)
    arr = np.zeros((1,))
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


# Precompute all drug fingerprints
print("Generating molecular fingerprints for usable drugs...")
fingerprints = {}
for dbid, row in tqdm(usable_drugs.iterrows(), total=len(usable_drugs)):
    smiles = row['smiles']
    fp = mol_to_fp(smiles)
    if fp is not None:
        fingerprints[dbid] = fp

print(f"✅ Generated fingerprints for {len(fingerprints)} drugs.")


Generating molecular fingerprints for usable drugs...


 14%|█▍        | 1740/12313 [00:01<00:05, 1781.44it/s][22:03:58] SMILES Parse Error: syntax error while parsing: OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(O)(O)O)C1=CC=C(C=C1)S([O-])([O-])[O-])\C1=CC=C(C=C1)S(O)(O)[O-]
[22:03:58] SMILES Parse Error: check for mistakes around position 84:
[22:03:58] C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(O
[22:03:58] ~~~~~~~~~~~~~~~~~~~~^
[22:03:58] SMILES Parse Error: extra open parentheses while parsing: OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(O)(O)O)C1=CC=C(C=C1)S([O-])([O-])[O-])\C1=CC=C(C=C1)S(O)(O)[O-]
[22:03:58] SMILES Parse Error: check for mistakes around position 40:
[22:03:58] 1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2
[22:03:58] ~~~~~~~~~~~~~~~~~~~~^
[22:03:58] SMILES Parse Error: extra open parentheses while parsing: OS(O)(O)C1=CC=C(C=C1)C-1=C2\C=CC(=N2)\C(=C2/N\C(\C=C2)=C(/C2=N/C(/C=C2)=C(\C2=CC=C\-1N2)C1=CC=C(C=C1)S(

✅ Generated fingerprints for 12303 drugs.


In [6]:
from tqdm import tqdm
import random

# --- Step 4: Create Training Pairs (Positive + Negative) ---

# (A) Build positive pairs and the positive set
positive_pairs = []
positive_set = set()
for _, row in tqdm(ddi_df.iterrows(), total=len(ddi_df)):
    a, b = row['drug_a_id'], row['drug_b_id']
    if a in fingerprints and b in fingerprints:
        pair = frozenset([a, b])
        if pair not in positive_set:
            positive_set.add(pair)
            positive_pairs.append((a, b, 1))

print(f"✅ Positive pairs collected: {len(positive_pairs)}")

# (B) Build negative pairs (random non-DDIs, not in positive set)
all_ids = list(fingerprints.keys())
negative_pairs = []
negative_needed = len(positive_pairs)

pbar = tqdm(total=negative_needed)
while len(negative_pairs) < negative_needed:
    a, b = random.sample(all_ids, 2)
    pair = frozenset([a, b])
    if pair not in positive_set:
        negative_pairs.append((a, b, 0))
        positive_set.add(pair)  # Prevent reuse as negative
        pbar.update(1)
pbar.close()

print(f"✅ Negative pairs generated: {len(negative_pairs)}")

# (C) Combine and shuffle
dataset = positive_pairs + negative_pairs
random.shuffle(dataset)
print(f"Training dataset size: {len(dataset)} (Positives: {len(positive_pairs)}, Negatives: {len(negative_pairs)})")


100%|██████████| 2323786/2323786 [02:28<00:00, 15604.31it/s]


✅ Positive pairs collected: 1161857


100%|██████████| 1161857/1161857 [00:06<00:00, 175680.55it/s]


✅ Negative pairs generated: 1161857
Training dataset size: 2323714 (Positives: 1161857, Negatives: 1161857)


In [7]:
import random

N = 100000  # choose the number based on your Colab RAM capacity
dataset = random.sample(dataset, N)
print(f"Using {N} pairs for ML (randomly sampled).")


Using 100000 pairs for ML (randomly sampled).


In [8]:

# --- Step 5: Feature Vector Construction ---

X, y = [], []
for a, b, label in tqdm(dataset):
    fp_a, fp_b = fingerprints[a], fingerprints[b]
    combined = np.concatenate([fp_a, fp_b])  # simple concatenation
    X.append(combined)
    y.append(label)

X = np.array(X)
y = np.array(y)
print(f"Feature matrix shape: {X.shape}")


100%|██████████| 100000/100000 [00:03<00:00, 25350.65it/s]


Feature matrix shape: (100000, 4096)


In [9]:
# --- Step 6: Train Random Forest Model ---

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
import joblib

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training Random Forest model...")
rf = RandomForestClassifier(
    n_estimators=200, max_depth=20, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

# Evaluate model
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("✅ Model Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification report:\n", classification_report(y_test, y_pred))


Training Random Forest model...
✅ Model Performance:
Accuracy: 0.8507
ROC AUC: 0.9261670809237768

Classification report:
               precision    recall  f1-score   support

           0       0.88      0.81      0.84      9843
           1       0.83      0.89      0.86     10157

    accuracy                           0.85     20000
   macro avg       0.85      0.85      0.85     20000
weighted avg       0.85      0.85      0.85     20000



In [10]:
# --- Step 7: Save Trained Model ---

joblib.dump(rf, "ddi_rf_model.pkl")
print("✅ Model saved as 'ddi_rf_model.pkl'")


✅ Model saved as 'ddi_rf_model.pkl'


In [ ]:
from google.colab import files
files.download("ddi_rf_model.pkl")


In [11]:
# --- Step 8: Test a Prediction ---

def predict_interaction_from_smiles(smiles_a, smiles_b):
    molA, molB = Chem.MolFromSmiles(smiles_a), Chem.MolFromSmiles(smiles_b)
    if molA is None or molB is None:
        return "Invalid SMILES input."
    fpA = mol_to_fp(smiles_a)
    fpB = mol_to_fp(smiles_b)
    features = np.concatenate([fpA, fpB]).reshape(1, -1)
    prob = rf.predict_proba(features)[0][1]
    risk = "High" if prob >= 0.75 else "Moderate" if prob >= 0.5 else "Low"
    print(f"Predicted Interaction Probability: {prob:.3f} → Risk Level: {risk}")

# Example test case: Warfarin & Simvastatin
warfarin = usable_drugs.loc["DB00682", "smiles"]
simvastatin = usable_drugs.loc["DB00641", "smiles"]
predict_interaction_from_smiles(warfarin, simvastatin)


Predicted Interaction Probability: 0.641 → Risk Level: Moderate
